# Glimpse3D — Serve Backend on Colab (GPU)

Run this notebook on a **GPU runtime** (Runtime → Change runtime type → GPU). It clones the repo, installs the backend dependencies, downloads the model weights (~6 GB), launches the FastAPI server on `0.0.0.0:8000`, and exposes it through a public Cloudflare tunnel. Copy the printed tunnel URL into your frontend's `.env` as `VITE_API_URL`, then run the frontend locally (`npm install && npm run dev`) on your own machine — it will talk to this GPU-backed API over the tunnel (HTTP and WebSocket / `wss` both work).

In [ ]:
# Confirm a GPU is attached. If this errors or shows no GPU, switch the runtime to GPU.
!nvidia-smi

In [ ]:
# Clone the repo and cd into it. If already cloned, the clone is skipped and we just cd in.
import os
if not os.path.isdir('/content/Glimpse3D/.git'):
    !git clone https://github.com/varunaditya27/Glimpse3D.git /content/Glimpse3D
%cd /content/Glimpse3D

In [ ]:
# Install dependencies. Colab already ships a CUDA-enabled torch, so we do NOT reinstall it here.
!pip install -r requirements-colab.txt
!pip install -r backend/requirements.txt
!pip install gdown huggingface_hub
# Cloudflare tunnel helper (wraps the cloudflared binary). If pip install fails,
# fall back to downloading the binary directly:
#   !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!pip install pycloudflared

In [ ]:
# Download model weights: SyncDreamer checkpoint + CLIP ViT-L-14, plus prewarm the HF caches
# (TripoSR, SDXL-Lightning, ControlNet-depth). ~6 GB total. Idempotent — safe to re-run; it
# skips files that are already present and valid.
!python scripts/download_weights.py --prewarm-hf

In [ ]:
# Launch the FastAPI backend in the background on 0.0.0.0:8000.
# main.py binds host 0.0.0.0 and reads PORT (defaults to 8000).
import subprocess, time, os

env = os.environ.copy()
# DEMO note: uncomment the next line ONLY for a no-GPU smoke test (stubs the heavy pipeline).
# env['GLIMPSE3D_DEMO_MODE'] = '1'

server = subprocess.Popen(
    ['python', '-m', 'backend.app.main'],
    cwd='/content/Glimpse3D',
    env=env,
)

# Give the server a moment to import models and bind the port.
time.sleep(20)
print('Backend launched (PID', server.pid, ') on http://localhost:8000')

In [ ]:
# Start a Cloudflare quick tunnel to the local backend and print the public https URL.
# Cloudflare quick tunnels forward WebSocket (wss) automatically, so the frontend's
# realtime /ws endpoint works through the same URL.
from pycloudflared import try_cloudflare

url = try_cloudflare(port=8000).tunnel
print('PUBLIC URL:', url)
print('Set VITE_API_URL in frontend/.env to the line above.')

# Fallback (pyngrok, needs a free authtoken from https://dashboard.ngrok.com):
#   !pip install pyngrok
#   from pyngrok import ngrok, conf
#   conf.get_default().auth_token = 'YOUR_NGROK_AUTHTOKEN'
#   url = ngrok.connect(8000, 'http').public_url
#   print('PUBLIC URL:', url)

## Next steps (run on your own machine)

1. In your local checkout, create / edit `frontend/.env` and set:
   ```
   VITE_API_URL=<the PUBLIC URL printed above>
   ```
2. Start the frontend:
   ```
   cd frontend
   npm install
   npm run dev
   ```
3. Open the app (Vite prints a local URL, usually http://localhost:5173). It now sends all API and WebSocket traffic to this GPU-backed Colab backend through the tunnel.

Keep this notebook tab running — the tunnel and the backend stay alive only while the Colab session is active.